# Approfondissement A2 — Gros volumes : lecture par blocs et agrégation incrémentale

## Le problème

Le fichier `nov.zip` de l'exercice Bstorm contient les événements d'un site de e-commerce pour un mois : 2,87 Go compressés, plusieurs dizaines de millions de lignes et plusieurs gigaoctets une fois décompressé. Il ne tient pas dans la mémoire d'un ordinateur portable.

Deux stratégies se complètent :

1. **réduire ce que l'on charge** : seulement les colonnes utiles, avec des types économes ;
2. **lire par blocs** (*chunks*) et combiner des résultats partiels, sans jamais avoir tout le fichier en mémoire.

Ce notebook utilise `nov_synthetique.zip` : 400 000 lignes générées avec **le même schéma** que le fichier Kaggle d'origine, pour pouvoir tout exécuter en classe. Les techniques s'appliquent telles quelles au vrai fichier.

In [ ]:
import numpy as np
import pandas as pd

FICHIER = "../donnees/chunk/nov_synthetique.zip"

## 1. Explorer sans tout charger

`nrows` lit les premières lignes : de quoi découvrir les colonnes et leurs types.

In [ ]:
apercu = pd.read_csv(FICHIER, nrows=1000)
apercu.head()

In [ ]:
apercu.dtypes

## 2. Charger moins : `usecols` et `dtype`

In [ ]:
complet = pd.read_csv(FICHIER)
print(round(complet.memory_usage(deep=True).sum() / 1e6, 1), "Mo pour toutes les colonnes")

reduit = pd.read_csv(FICHIER, usecols=["event_type", "category_code", "price"],
                     dtype={"event_type": "category", "category_code": "category", "price": "float32"})
print(round(reduit.memory_usage(deep=True).sum() / 1e6, 1), "Mo avec 3 colonnes et des types adaptés")

Avant de lire par blocs, on se demande toujours si cette réduction ne suffit pas.

## 3. Lire par blocs : `chunksize`

Avec `chunksize`, `read_csv` ne renvoie pas un DataFrame mais un **itérateur** : chaque tour de boucle produit un DataFrame de `chunksize` lignes.

In [ ]:
lecteur = pd.read_csv(FICHIER, chunksize=100_000, usecols=["event_type", "price"])
type(lecteur)

In [ ]:
for i, bloc in enumerate(lecteur):
    print(i, bloc.shape, bloc.index[0], "→", bloc.index[-1])

L'index continue d'un bloc à l'autre. Une fois parcouru, l'itérateur est **épuisé** : une seconde boucle sur `lecteur` ne produirait rien. Il faut le recréer (ou l'ouvrir dans un `with`).

> **Prédiction.** Que produit la cellule suivante ?

In [ ]:
sum(1 for _ in lecteur)

## 4. Agréger par blocs

Le principe est celui du *map-reduce* :

1. sur chaque bloc, calculer des **quantités additives** : un nombre de lignes, une somme, une somme des carrés ;
2. additionner ces quantités entre les blocs ;
3. en déduire les statistiques finales (moyenne, variance).

Une moyenne ne s'additionne pas, mais une somme et un effectif si : moyenne = somme / effectif.

In [ ]:
partiels = []
with pd.read_csv(FICHIER, chunksize=100_000, usecols=["event_type", "price"]) as lecteur:
    for bloc in lecteur:
        g = bloc.groupby("event_type")["price"]
        partiels.append(pd.DataFrame({
            "n": g.count(),
            "n_nan": g.size() - g.count(),
            "somme": g.sum(),
            "min": g.min(),
            "max": g.max(),
        }))

partiels[0]

On combine les résultats partiels : somme pour les compteurs et les sommes, minimum et maximum pour les extrêmes.

In [ ]:
tous = pd.concat(partiels)
total = tous.groupby(level=0).agg({"n": "sum", "n_nan": "sum", "somme": "sum", "min": "min", "max": "max"})
total["moyenne"] = total["somme"] / total["n"]
total

Vérification avec le calcul direct (possible ici, car le fichier synthétique tient en mémoire) :

In [ ]:
complet.groupby("event_type")["price"].agg(["count", "mean", "min", "max"])

### Combiner des comptages : `add(fill_value=0)`

Pour une variable catégorielle, chaque bloc produit un `value_counts`. Une modalité peut être absente d'un bloc : on additionne avec `fill_value=0` (chapitre 1).

In [ ]:
comptes = None
with pd.read_csv(FICHIER, chunksize=100_000, usecols=["category_code"]) as lecteur:
    for bloc in lecteur:
        vc = bloc["category_code"].value_counts(dropna=False)
        comptes = vc if comptes is None else comptes.add(vc, fill_value=0)
comptes.astype(int).sort_values(ascending=False)

## 5. La variance par blocs : attention à la stabilité numérique

L'énoncé Bstorm propose la formule :

$$\mathrm{Var}(x) = \frac{1}{N}\sum x_i^2 - \bar{x}^2$$

Elle est exacte en mathématiques, et additive (il suffit de cumuler N, la somme et la somme des carrés). Mais en calcul flottant, elle soustrait deux grands nombres presque égaux : une **annulation catastrophique**.

> **Pourquoi ?** Les valeurs ci-dessous sont des prix autour de 100 millions avec une variation de quelques unités. Comparez la formule de l'énoncé et le calcul de Pandas.

In [ ]:
rng = np.random.default_rng(0)
x = pd.Series(1e8 + rng.random(1_000_000))
naive = (x ** 2).sum() / len(x) - x.mean() ** 2
print("formule naïve :", naive)
print("Pandas        :", x.var(ddof=0))

La vraie variance d'une loi uniforme sur [0, 1] vaut 1/12 ≈ 0,083. La formule naïve donne un résultat sans aucun rapport, parfois négatif. Sur des prix de quelques centaines d'euros, l'erreur est faible, mais la méthode reste fragile.

Une solution simple et additive : **centrer les valeurs** sur une constante proche de la moyenne (par exemple la moyenne du premier bloc) avant de cumuler les sommes. La variance ne change pas quand on décale toutes les valeurs.

In [ ]:
decalage = x.iloc[:1000].mean()
y = x - decalage
centre = (y ** 2).sum() / len(y) - y.mean() ** 2
print("formule centrée :", centre)

Pour l'exercice, on peut aussi combiner, bloc par bloc, les effectifs, les moyennes et les variances de chaque bloc avec la formule de Chan, utilisée par les bibliothèques statistiques. Le corrigé compatible Pandas 3 (`corrections/bstorm/pandas3/05_chunk_processing/`) utilise le centrage.

## 6. Discrétiser par blocs

Pour étudier le prix en fonction du type d'événement, on découpe les prix en tranches fixes. Les bornes doivent être **les mêmes pour tous les blocs**, sinon les tranches ne sont pas comparables : on les fixe avant la boucle.

In [ ]:
bornes = [0, 50, 100, 200, 500, 1000, np.inf]
tableau = None
with pd.read_csv(FICHIER, chunksize=100_000, usecols=["event_type", "price"]) as lecteur:
    for bloc in lecteur:
        tranche = pd.cut(bloc["price"], bins=bornes)
        t = pd.crosstab(tranche, bloc["event_type"])
        tableau = t if tableau is None else tableau.add(t, fill_value=0)
tableau.astype(int)

Pour plusieurs statistiques à la fois, `pivot_table` accepte une liste de fonctions, y compris une fonction personnelle. La correction Bstorm cumule ainsi l'effectif, la somme et la somme des carrés par combinaison (type d'événement, tranche de prix) :

In [ ]:
def somme_carres(s):
    return (s ** 2).sum()

bloc = pd.read_csv(FICHIER, nrows=100_000, usecols=["event_type", "price"])
bloc["tranche"] = pd.cut(bloc["price"], bins=bornes)
bloc.pivot_table(index=["event_type", "tranche"], values="price",
                 aggfunc=["count", "sum", somme_carres], observed=True).head()

Ces trois quantités sont additives : les tableaux de chaque bloc se combinent avec `add(fill_value=0)`, comme ci-dessus.

`pd.cut` laisse les prix manquants en `NaN`, au lieu de les convertir en un entier absurde. La correction d'origine calculait la tranche avec `np.int32(...)` : un `NaN` y devient silencieusement -2147483648.

In [ ]:
np.array([np.nan, 120.0]).astype(np.int32)

## 7. Écrire par blocs, et un format plus efficace

On peut aussi transformer le fichier bloc par bloc et écrire le résultat au fur et à mesure (`mode="a"` pour ajouter à un CSV existant). Pour des données destinées à être relues souvent, le format **Parquet** est bien plus efficace que le CSV : stockage par colonnes, types conservés, compression. Il nécessite `pyarrow`, installé avec le projet.

In [ ]:
reduit.to_parquet("../sorties/nov_reduit.parquet", index=False)
import os
print(round(os.path.getsize("../sorties/nov_reduit.parquet") / 1e6, 1), "Mo en Parquet")
pd.read_parquet("../sorties/nov_reduit.parquet", columns=["price"]).shape

Au-delà de ce que permettent les blocs, d'autres outils prennent le relais : Polars, DuckDB, Dask ou Spark, qui reprennent largement les concepts de Pandas.

## Exercice

- Exercice Bstorm **chunk processing** (`exercices/bstorm/05_chunk_processing/`). L'énoncé fait référence à `nov.zip` (2,87 Go, à télécharger sur Kaggle) ; le fichier `donnees/chunk/nov_synthetique.zip` a le même schéma et permet de travailler sans téléchargement.

## À retenir

- Réduire d'abord : `usecols`, `dtype`, `nrows`.
- `chunksize` renvoie un itérateur de DataFrame, à n'utiliser qu'une fois (ou dans un `with`).
- On cumule des quantités additives (effectifs, sommes), puis on en déduit les statistiques.
- `add(fill_value=0)` combine des comptages partiels.
- La formule Σx²/N − moyenne² est instable : centrer les valeurs avant de cumuler.
- Bornes de discrétisation fixées une fois pour tous les blocs ; `pd.cut` gère les `NaN`.